# Building a Semantic Search Engine

This notebook walks through building a semantic search engine from scratch:
1. Generate embeddings for a document collection
2. Store them in Pinecone
3. Query with natural language
4. Filter by metadata
5. Compare keyword vs semantic search

## Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY not found in .env")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API keys loaded.")

In [ ]:
from pinecone import Pinecone, ServerlessSpec
from openai import OpenAI

pc = Pinecone(api_key=PINECONE_API_KEY)
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Embedding model settings
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

## Document Collection

We'll use a small collection of technology-related documents.

In [ ]:
documents = [
    {
        "id": "doc1",
        "text": "Python is a versatile programming language widely used in data science, machine learning, and web development. Its simple syntax makes it great for beginners.",
        "metadata": {
            "topic": "programming",
            "language": "Python",
            "difficulty": "beginner"
        }
    },
    {
        "id": "doc2",
        "text": "Deep learning models like transformers have revolutionized natural language processing. GPT and BERT are two of the most influential architectures.",
        "metadata": {
            "topic": "deep-learning",
            "language": "English",
            "difficulty": "advanced"
        }
    },
    {
        "id": "doc3",
        "text": "React is a JavaScript library for building user interfaces. It uses a component-based architecture and virtual DOM for efficient rendering.",
        "metadata": {
            "topic": "web-development",
            "language": "JavaScript",
            "difficulty": "intermediate"
        }
    },
    {
        "id": "doc4",
        "text": "PostgreSQL is an open-source relational database known for its reliability and extensibility. It supports advanced SQL features and JSON data.",
        "metadata": {
            "topic": "databases",
            "language": "SQL",
            "difficulty": "intermediate"
        }
    },
    {
        "id": "doc5",
        "text": "Docker containers package applications with their dependencies, ensuring consistent deployment across development and production environments.",
        "metadata": {
            "topic": "devops",
            "language": "YAML",
            "difficulty": "intermediate"
        }
    },
    {
        "id": "doc6",
        "text": "Vector databases store high-dimensional embeddings and enable fast similarity search. They are essential for building semantic search and recommendation systems.",
        "metadata": {
            "topic": "databases",
            "language": "Python",
            "difficulty": "advanced"
        }
    },
    {
        "id": "doc7",
        "text": "Kubernetes orchestrates containerized applications across clusters of machines. It handles scaling, deployment, and self-healing of services.",
        "metadata": {
            "topic": "devops",
            "language": "YAML",
            "difficulty": "advanced"
        }
    },
    {
        "id": "doc8",
        "text": "Rust is a systems programming language focused on safety and performance. It prevents memory bugs at compile time without garbage collection.",
        "metadata": {
            "topic": "programming",
            "language": "Rust",
            "difficulty": "advanced"
        }
    }
]

print(f"Loaded {len(documents)} documents.")

## Generate Embeddings

Convert each document's text into a vector using OpenAI's embedding model.

In [ ]:
def get_embeddings(texts: list[str], model: str = EMBEDDING_MODEL) -> list[list[float]]:
    """Generate embeddings for a list of texts."""
    response = openai_client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]


# Generate embeddings for all documents
texts = [doc["text"] for doc in documents]
embeddings = get_embeddings(texts)

print(f"Generated {len(embeddings)} embeddings.")
print(f"Embedding dimension: {len(embeddings[0])}")

## Store in Pinecone

Create an index and upsert the document embeddings with their metadata.

In [ ]:
INDEX_NAME = "semantic-search-demo"

# Clean up if exists
if INDEX_NAME in pc.list_indexes().names:
    pc.delete_index(INDEX_NAME)

# Create index
pc.create_index(
    name=INDEX_NAME,
    dimension=EMBEDDING_DIM,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

index = pc.Index(INDEX_NAME)
print(f"Created index: {INDEX_NAME}")

In [ ]:
# Prepare vectors for upsert
vectors_to_upsert = [
    {
        "id": doc["id"],
        "values": emb,
        "metadata": doc["metadata"] | {"text": doc["text"]}  # Merge metadata with text
    }
    for doc, emb in zip(documents, embeddings)
]

# Upsert
index.upsert(vectors=vectors_to_upsert)
print(f"Upserted {len(vectors_to_upsert)} vectors.")
print(f"Index stats: {index.describe_index_stats()}")

## Query with Natural Language

Convert a natural language query to an embedding, then search Pinecone.

In [ ]:
def semantic_search(query: str, top_k: int = 3, filter_dict: dict = None) -> list[dict]:
    """Search for documents similar to the query."""
    # Embed the query
    query_embedding = get_embeddings([query])[0]
    
    # Build query parameters
    query_params = {
        "vector": query_embedding,
        "top_k": top_k,
        "include_metadata": True
    }
    if filter_dict:
        query_params["filter"] = filter_dict
    
    # Search
    results = index.query(**query_params)
    
    return [
        {
            "id": match["id"],
            "score": match["score"],
            "text": match["metadata"]["text"],
            "metadata": {k: v for k, v in match["metadata"].items() if k != "text"}
        }
        for match in results["matches"]]


def display_results(results: list[dict], query: str):
    """Display search results in a readable format."""
    print(f"Query: \"{query}\"")
    print("=" * 70)
    for i, r in enumerate(results, 1):
        print(f"{i}. [{r['id']}] Score: {r['score']:.4f}")
        print(f"   {r['text']}")
        print(f"   Topic: {r['metadata']['topic']} | Difficulty: {r['metadata']['difficulty']}")
        print()

In [ ]:
# Search 1: General query
results = semantic_search("What language is good for data analysis?")
display_results(results, "What language is good for data analysis?")

In [ ]:
# Search 2: Technical query
results = semantic_search("How to deploy applications in production?")
display_results(results, "How to deploy applications in production?")

In [ ]:
# Search 3: Conceptual query (no exact keyword match in documents)
results = semantic_search("teaching machines to understand human language")
display_results(results, "teaching machines to understand human language")

## Filtering by Metadata

Combine semantic search with metadata filters for precise results.

In [ ]:
# Search only in programming topics
results = semantic_search(
    "Which language should I learn?",
    filter_dict={"topic": {"$eq": "programming"}}
)
display_results(results, "Which language should I learn? (filtered: programming topic)")

In [ ]:
# Search for beginner-friendly content only
results = semantic_search(
    "getting started with technology",
    filter_dict={"difficulty": {"$eq": "beginner"}}
)
display_results(results, "getting started with technology (filtered: beginner only)")

In [ ]:
# Combined filter: databases + advanced
results = semantic_search(
    "storing and querying data efficiently",
    filter_dict={
        "$and": [
            {"topic": {"$eq": "databases"}},
            {"difficulty": {"$eq": "advanced"}}
        ]
    }
)
display_results(results, "storing and querying data efficiently (filtered: databases + advanced)")

## Keyword Search vs Semantic Search

Compare how keyword matching and semantic search handle the same queries.

In [ ]:
import re


def keyword_search(query: str, documents: list[dict], top_k: int = 3) -> list[dict]:
    """Simple keyword search using token overlap (TF-inspired scoring)."""
    query_tokens = set(re.findall(r'\w+', query.lower()))
    
    scored = []
    for doc in documents:
        doc_tokens = set(re.findall(r'\w+', doc["text"].lower()))
        # Score = number of matching tokens / total query tokens
        overlap = query_tokens & doc_tokens
        score = len(overlap) / len(query_tokens) if query_tokens else 0
        scored.append({"id": doc["id"], "score": score, "text": doc["text"], "metadata": doc["metadata"]})
    
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


def display_comparison(query: str, top_k: int = 3):
    """Compare keyword and semantic search results."""
    kw_results = keyword_search(query, documents, top_k)
    sem_results = semantic_search(query, top_k)
    
    print(f"Query: \"{query}\"")
    print("=" * 70)
    
    print("\nKeyword Search Results:")
    print("-" * 70)
    for i, r in enumerate(kw_results, 1):
        print(f"  {i}. [{r['id']}] Score: {r['score']:.4f} | {r['text'][:80]}...")
    
    print("\nSemantic Search Results:")
    print("-" * 70)
    for i, r in enumerate(sem_results, 1):
        print(f"  {i}. [{r['id']}] Score: {r['score']:.4f} | {r['text'][:80]}...")
    print()

In [ ]:
# Comparison 1: Query with no exact keyword matches
display_comparison("systems programming without memory safety issues")

In [ ]:
# Comparison 2: Query with some keyword overlap
display_comparison("JavaScript web application development")

In [ ]:
# Comparison 3: Conceptual query with paraphrasing
display_comparison("how to put software into production safely")

## Cleanup

In [ ]:
pc.delete_index(INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")

## Exercises

### Exercise 1: Expand the Document Collection

1. Add 10+ more documents covering different topics (AI, cybersecurity, cloud computing, etc.)
2. Embed and upsert them into the index
3. Run queries to verify they appear in relevant results

### Exercise 2: Multi-Filter Search

1. Create a search function that accepts multiple filter criteria
2. Test with combinations: topic=programming AND difficulty=advanced
3. Test with OR filters: topic=programming OR topic=deep-learning

### Exercise 3: Search Quality Evaluation

1. Write 10 test queries with expected document IDs
2. Run both keyword and semantic search
3. Calculate precision@3 for each method
4. Which method performs better? Why?

### Exercise 4: RAG Pipeline

1. Take the top search result's text
2. Pass it as context to an LLM (OpenAI chat completion)
3. Generate a response to the original query
4. This is the foundation of Retrieval-Augmented Generation (RAG)

## Key Takeaways

- Semantic search finds relevant results even without keyword overlap
- Metadata filtering narrows results while preserving semantic ranking
- Keyword search struggles with synonyms and paraphrasing
- Combining semantic search with LLMs creates powerful RAG systems